# Embeddings & Vector Store

Build a FAISS index over the combined KB (256,841 records from 4 sources).

Input:
- `data/processed/kb_combined.csv`

Outputs:
- `data/embeddings/faiss_index/kb_index_flatip.faiss`
- `data/embeddings/faiss_index/kb_chunk_mapping.pkl`

One thing to watch while splitting: every chunk keeps a copy of its record's
question, so a chunk whose body is empty is just the question matching itself.
Those chunks are dropped before encoding.


In [13]:
import os
import pickle
import re
import time
from pathlib import Path

import torch
import faiss
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from tqdm import tqdm

ROOT = Path.cwd().parent
PROCESSED_DIR = ROOT / "data" / "processed"
INDEX_DIR = ROOT / "data" / "embeddings" / "faiss_index"
INDEX_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


Device: cuda


## 1. Load KB

Four sources contribute to the knowledge base:

| Source | Records | Has context? | Has category? |
|--------|---------|--------------|----------------|
| pubmedqa | 211,267 | yes | no |
| medredqa | 29,581 | yes | no |
| medquad | 14,978 | no | yes |
| medlineplus | 1,015 | no | no |


In [14]:
kb = pd.read_csv(PROCESSED_DIR / "kb_combined.csv", low_memory=False)
kb["context"] = kb["context"].fillna("")
kb["answer"] = kb["answer"].fillna("")
kb["category"] = kb["category"].fillna("")

print("kb_combined:", kb.shape)
print(kb["source"].value_counts())
print(kb.columns.tolist())


kb_combined: (256841, 7)
source
pubmedqa       211267
medredqa        29581
medquad         14978
medlineplus      1015
Name: count, dtype: int64
['doc_id', 'source', 'source_id', 'category', 'question', 'context', 'answer']


## 2. Build chunks

Some sources (especially MedRedQA) contain decorative character runs like `_____` and `|||||` that inflate token counts without adding meaning. We strip them before splitting.

Splitter settings: chunk_size=1000, overlap=150.

The question is prepended to **every** chunk of a record — not just the first one. This helps retrieval, since a middle chunk that only contains a fragment of the context may not contain the question's keywords.

The splitter can still return a chunk that holds nothing but the prepended question: when the piece after the question is longer than `chunk_size`, the question line is flushed on its own first. That chunk carries no fact, and because it is a near-copy of the question it scores very high against any question phrased the same way. Left in, it fills the top of the ranking with empty results, so chunks with less than 30 characters after the question line are dropped here.

Two text formats:
- PubMedQA + MedRedQA: `Question + Context + Answer`
- MedQuAD + MedlinePlus: `Question + Answer` (no context field)


In [15]:
MIN_CONTENT_CHARS = 30


def clean_text(t):
    # collapse runs of decorative characters
    t = re.sub(r"_{3,}", " ", t)
    t = re.sub(r"-{3,}", " ", t)
    t = re.sub(r"={3,}", " ", t)
    t = re.sub(r"\*{3,}", " ", t)
    # normalize markdown table pipes
    t = re.sub(r"\s*\|\s*", " | ", t)
    # collapse multiple spaces
    t = re.sub(r"[ \t]{2,}", " ", t)
    return t.strip()


def content_after_question(chunk):
    """Text left once the prepended `Question: ...` line is removed."""
    if chunk.startswith("Question:"):
        newline = chunk.find("\n")
        return "" if newline == -1 else chunk[newline + 1:]
    return chunk


splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""],
)

all_chunks = []
all_meta = []
dropped = 0

for _, row in kb.iterrows():
    q = clean_text(row["question"])
    c = clean_text(row["context"])
    a = clean_text(row["answer"])

    if c:
        full_text = f"Question: {q}\nContext: {c}\nAnswer: {a}"
    else:
        full_text = f"Question: {q}\nAnswer: {a}"

    for chunk in splitter.split_text(full_text):
        # Add the question to every chunk so middle chunks keep the topic
        if q and not chunk.startswith("Question:"):
            chunk = f"Question: {q}\n{chunk}"

        # Skip chunks that are only the question — they hold no fact
        if len(content_after_question(chunk)) < MIN_CONTENT_CHARS:
            dropped += 1
            continue

        all_chunks.append(chunk)
        all_meta.append({
            "doc_id": row["doc_id"],
            "source": row["source"],
            "source_id": row["source_id"],
            "category": row["category"],
            "text_chunk": chunk,
        })

df_chunks = pd.DataFrame(all_meta)
text_chunks = all_chunks
n_chunks = len(text_chunks)
n_records = len(kb)

# The two lists are no longer needed; only the DataFrame and the text list are.
del all_chunks, all_meta

print(f"Built {n_chunks:,} chunks from {n_records:,} records")
print(f"Dropped {dropped:,} chunks with no text after the question "
      f"({dropped / (n_chunks + dropped):.1%})")
print()
print("Chunks per source:")
print(df_chunks["source"].value_counts())
print()
print(f"Chunks per record: {n_chunks / n_records:.2f}")
print()
print("Sample chunk 1:")
print(text_chunks[0][:300])
print()
print("Sample chunk 2:")
print(text_chunks[1][:300])

del kb  # free memory before encoding


Built 727,138 chunks from 256,841 records
Dropped 219,458 chunks with no text after the question (23.2%)

Chunks per source:
source
pubmedqa       624396
medredqa        70716
medquad         29181
medlineplus      2845
Name: count, dtype: int64

Chunks per record: 2.83

Sample chunk 1:
Question: What are the treatments for Breast Cancer?
Answer: Key Points - There are different types of treatment for patients with breast cancer. - Five types of standard treatment are used: - Surgery - Radiation therapy - Chemotherapy - Hormone therapy - Targeted therapy - New types of treatment ar

Sample chunk 2:
Question: What are the treatments for Breast Cancer?
. When clinical trials show that a new treatment is better than the standard treatment, the new treatment may become the standard treatment. Patients may want to think about taking part in a clinical trial. Some clinical trials are open only to pa


## 3. Load embedding model

`BAAI/bge-m3` — multilingual embedding model, 1024-dim output. fp16 on GPU to speed things up and to keep the model inside 8 GB of VRAM.

`max_seq_length` is raised in the next section once we know how long the chunks actually are; bge-m3 does not need an instruction prefix on the query side.


In [ ]:
MODEL_NAME = "BAAI/bge-m3"

model = SentenceTransformer(MODEL_NAME, device=DEVICE)
model.max_seq_length = 512

if DEVICE == "cuda":
    model = model.half()

print("Loaded:", MODEL_NAME)
print("Dimension:", model.get_sentence_embedding_dimension())
print("Max seq length:", model.max_seq_length)


## 4. Check chunk lengths before encoding

Run the tokenizer over all chunks and check whether anything exceeds `max_seq_length`. Catching truncation here saves hours of wasted encoding.


In [ ]:
def tokenize_lengths(texts, tokenizer, batch_size=2000):
    lengths = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Tokenizing"):
        batch = texts[i:i + batch_size]
        enc = tokenizer(batch, add_special_tokens=True)["input_ids"]
        lengths.extend(len(e) for e in enc)
    return np.array(lengths)


token_lengths = tokenize_lengths(text_chunks, model.tokenizer)

print("Max tokens:", token_lengths.max())
print("Mean tokens:", round(token_lengths.mean(), 1))
print("P99 tokens:", int(np.percentile(token_lengths, 99)))
print("Chunks > 512:", (token_lengths > 512).sum())

if token_lengths.max() > model.max_seq_length:
    model.max_seq_length = int(token_lengths.max()) + 64
    print("Raised max_seq_length to", model.max_seq_length)
else:
    print("All chunks fit in", model.max_seq_length, "tokens")


In [ ]:
# Check the chunks that are longer than 512 tokens
long_idx = np.where(token_lengths > 512)[0]

print(f"{len(long_idx)} chunks over 512 tokens:")
print()
for idx in long_idx:
    src = df_chunks.iloc[idx]["source"]
    print(f"[{idx}] {src} | {token_lengths[idx]} tokens")
    print(f"    {text_chunks[idx][:180]}...")
    print()


## 5. Encode chunks

Normalized embeddings → cosine similarity via inner product.

Batch size is 32: big enough to keep the GPU busy, small enough that the tokenizer stays the bottleneck.

All 727,138 kept chunks are encoded in one pass on the GPU in fp16. On the reference machine this took about 1h 50m at ~9 ms per chunk. The saved index is derived directly from these vectors — no separate rebuild step.


In [ ]:
BATCH_SIZE = 32

# Preallocate instead of growing a list + vstack
embeddings = np.empty(
    (n_chunks, model.get_sentence_embedding_dimension()),
    dtype=np.float32,
)

n_batches = (n_chunks + BATCH_SIZE - 1) // BATCH_SIZE
print(f"Encoding {n_chunks:,} chunks in {n_batches} batches (batch_size={BATCH_SIZE})")
start = time.time()

for i in tqdm(range(0, n_chunks, BATCH_SIZE), desc="Encoding", unit="batch"):
    batch = text_chunks[i:i + BATCH_SIZE]
    emb = model.encode(
        batch,
        batch_size=BATCH_SIZE,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    embeddings[i:i + len(batch)] = emb

    processed = min(i + BATCH_SIZE, n_chunks)
    if processed % 50000 == 0 or processed == n_chunks:
        print(f"  {processed:,} / {n_chunks:,}")

elapsed = time.time() - start

print()
print("Shape:", embeddings.shape)
print("Dtype:", embeddings.dtype)
print(f"Time: {elapsed:.1f}s ({elapsed / n_chunks * 1000:.1f}ms per chunk)")


## 6. Build FAISS index

`IndexFlatIP` — exact search with inner product (= cosine similarity because embeddings are normalized).


In [ ]:
d = embeddings.shape[1]

index = faiss.IndexFlatIP(d)
index.add(embeddings)

print("Dimension:", d)
print("Vectors:", f"{index.ntotal:,}")


## 7. Save index and chunk mapping

The mapping keeps only the chunk-level columns: `chunk_id`, `doc_id`, `source`, `source_id`, `category`, `text_chunk`.

`question`, `context` and `answer` are not copied here. They are identical for every chunk of a record, and duplicating them across 800k rows is what pushed the previous pickle to 973 MB and the CSV to 2.2 GB. Join them back from `kb_combined.csv` on `doc_id` when needed.


In [ ]:
index_path = INDEX_DIR / "kb_index_flatip.faiss"
mapping_pkl_path = INDEX_DIR / "kb_chunk_mapping.pkl"

faiss.write_index(index, str(index_path))

mapping_df = df_chunks.copy()
mapping_df.insert(0, "chunk_id", np.arange(len(mapping_df), dtype=np.int32))

with open(mapping_pkl_path, "wb") as f:
    pickle.dump(mapping_df, f, protocol=4)

print(f"FAISS index: {index_path}")
print(f"  Vectors: {index.ntotal:,}")
print(f"  File size: {os.path.getsize(index_path) / 1024**2:.1f} MB")
print(f"Chunk mapping: {mapping_pkl_path}")
print(f"  Rows: {len(mapping_df):,}")
print(f"  Columns: {mapping_df.columns.tolist()}")
print(f"  File size: {os.path.getsize(mapping_pkl_path) / 1024**2:.1f} MB")

assert index.ntotal == len(mapping_df), (
    f"Mismatch: index={index.ntotal}, mapping={len(mapping_df)}"
)

print()
print(f"Sanity check passed: {index.ntotal:,} vectors == {len(mapping_df):,} mapping rows")


## 8. Upload to HuggingFace (optional)

The saved files are pushed to the project's HuggingFace dataset so fresh clones can auto-download them.

**This cell performs an actual upload of ~3.3 GB. Skip it (just do not run it) unless you have decided the index shape is final and want to publish.**

If you use "Run All", this cell will run. If you want to run the notebook without uploading, run the cells one by one and stop before this one, or comment the `upload_file` calls out.


In [ ]:
# OPTIONAL — only run this cell if you want to publish the index to HuggingFace.
# Requires HF_TOKEN (or a logged-in huggingface_hub session) in the environment.

import sys
sys.path.append(os.path.abspath(".."))

from dotenv import load_dotenv
load_dotenv()

from src.data.hub import upload_file

print("Syncing to HuggingFace...")

upload_file(
    "data/embeddings/faiss_index/kb_index_flatip.faiss",
    "embeddings/kb_index_flatip.faiss",
)

upload_file(
    "data/embeddings/faiss_index/kb_chunk_mapping.pkl",
    "embeddings/kb_chunk_mapping.pkl",
)


## 9. Sanity-check retrieval

Five queries, top-3 each. Two goals:
- Verify the index returns relevant chunks
- Measure query latency (FAISS search only, encoding excluded)


In [ ]:
test_queries = [
    "What are the symptoms of type 2 diabetes?",
    "How is pneumonia diagnosed in elderly patients?",
    "What are the side effects of metformin?",
    "How can cardiovascular disease be prevented?",
    "What is the role of the immune system in fighting infections?",
]

query_embeddings = model.encode(
    test_queries,
    batch_size=16,
    show_progress_bar=False,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

query_embeddings = np.asarray(query_embeddings, dtype=np.float32)

k = 3
latencies = []

for qi, query in enumerate(test_queries):
    t0 = time.perf_counter()
    D, I = index.search(query_embeddings[qi:qi + 1], k)
    latency_ms = (time.perf_counter() - t0) * 1000
    latencies.append(latency_ms)

    print("=" * 100)
    print(f"Query {qi + 1}: {query}")
    print(f"Latency: {latency_ms:.2f}ms")
    print()

    for rank in range(k):
        idx = int(I[0, rank])
        score = float(D[0, rank])
        row = mapping_df.iloc[idx]

        print(
            f"Top {rank + 1} | chunk {idx} | "
            f"source: {row['source']} | score: {score:.4f}"
        )
        print(f"{row['text_chunk'][:200]}")
        print("-" * 100)

print()
print("Latency summary")
print(f"Min: {min(latencies):.2f}ms")
print(f"Max: {max(latencies):.2f}ms")
print(f"Mean: {np.mean(latencies):.2f}ms")
print(f"Median: {np.median(latencies):.2f}ms")
print()

if max(latencies) < 500:
    print("KPI MET: all queries returned in < 500ms")
else:
    print("KPI NOT MET: some queries exceeded 500ms")


## Summary

| Item | Value |
|---|---|
| Records in KB | 256,841 (4 sources) |
| Chunks built | 727,138 (219,458 dropped as question-only) |
| Chunk size | 1000 chars, 150 overlap |
| Question prepended | yes (every chunk) |
| Chunks per record | 2.83 overall, 2.96 for PubMedQA |
| Embedding model | `BAAI/bge-m3` (1024d) |
| max_seq_length | 858 (longest chunk is 794 tokens) |
| FAISS index type | `IndexFlatIP` |
| Index size | 2,840 MB |
| Mapping columns | `chunk_id, doc_id, source, source_id, category, text_chunk` |
| Sanity check | 5 queries x top-3 |
| Latency KPI | < 500ms per query |

Files saved:
- `data/embeddings/faiss_index/kb_index_flatip.faiss`
- `data/embeddings/faiss_index/kb_chunk_mapping.pkl`

Next: `06_rag_pipeline.ipynb`
